# UC myeloid cells: selenium-influenced upstream regulators (Fig 2G)

Stacked violin of selenium-influenced upstream regulator genes (SELENOP, IL1B, TREM1, TNF, IL1A, IL6, STAT3, NFKB1, RELA)
across macrophage subtypes of the UC atlas.

**Inputs**: `uc_atlas_annotated.h5ad` (06_atlas_assembly.R); myeloid UMAP and Harmony embedding exported by
`03_myeloid_subclustering.R`.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import palantir
from matplotlib.colors import LinearSegmentedColormap
from cytotrace2_py.cytotrace2_py import cytotrace2
import fast_matrix_market as fmm
import scipy.sparse

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/uc/scrna/output"
MYE_DIR    = f"{DATA_DIR}/myeloid"
OUTPUT_DIR = f"{DATA_DIR}/myeloid"

## 1. Load myeloid cells and attach embeddings

In [ ]:
# Load full atlas and subset to myeloid
scrna = sc.read_h5ad(f"{DATA_DIR}/uc_atlas_annotated.h5ad")
mye   = scrna[scrna.obs["cell_category"] == "Myeloid"].copy()

print(mye.obs["cell_type_short"].value_counts())

In [ ]:
# Attach myeloid UMAP coordinates (exported from R subclustering)
umap_mye = pd.read_csv(f"{MYE_DIR}/umap_myeloid_iter3_anno.csv")
mye.obsm["X_umap"] = umap_mye[["umapharmonymyeiter3_1",
                                  "umapharmonymyeiter3_2"]].to_numpy()

# Attach Harmony embedding
harm = pd.read_csv(f"{MYE_DIR}/mye_harmony_iter3.csv", index_col=0)
mye.obsm["X_harmony"] = harm.to_numpy()

In [ ]:
# Exclude cycling, neutrophils, and rare DC populations for trajectory analysis
exclude = ["Cycl Myeloid", "Neutrophil", "cDC1", "CCR7+ Mat DC"]
mye = mye[~mye.obs["cell_type_short"].isin(exclude)].copy()

print(mye.obs["cell_type_short"].value_counts())

## 5. Selenium-influenced upstream regulators (Fig 2G)

Stacked violin of SE-affected upstream regulator genes across macrophage subtypes.

In [ ]:
inhibited_gene_symbols = ['SELENOP', "IL1B", "TREM1", "TNF", "IL1A", "IL6", "STAT3", "NFKB1", "RELA"]

# Ordered macrophage subtypes
desired_order = ['Mo-Mac', 'Trans cDC2/Mac', 'Inf Mo-Mac', 'Mac S+M+S+',
                 'Mac S+SG+', 'Mac S+M+P+', 'Mac S+XS-']

scrna_mac = mye[mye.obs['cell_type_short'].isin(desired_order)].copy()
scrna_mac.obs['cell_type_short'] = pd.Categorical(
    scrna_mac.obs['cell_type_short'],
    categories=desired_order,
    ordered=True
)

# Recompute dendrogram with optimal leaf ordering
sc.tl.dendrogram(scrna_mac, groupby='cell_type_short', optimal_ordering=True)

sc.pl.stacked_violin(
    scrna_mac,
    var_names=inhibited_gene_symbols[:7],
    groupby='cell_type_short',
    swap_axes=False,
    dendrogram=True,
    figsize=(8, 4),
    cmap='plasma',
    title="",
    show=False
)
fig = plt.gcf()

for ax in fig.axes:
    for label in ax.get_xticklabels():
        label.set_rotation(0)

fig.text(0.4, 0.02, "SE-affected Upstream Regulators", ha='center', va='center', fontsize=10)
fig.text(-0.008, 0.5, "Cell Type", ha='center', va='center', rotation='vertical', fontsize=10)
plt.tight_layout(rect=[0.2, 0.2, 1, 0.9])
plt.show()